### Set up Ollama in Colab

To make `langchain_ollama.ChatOllama` work, we need an Ollama server running and the `llama3.1:8b` model available on it. Since Colab runs on a remote VM, we'll install Ollama directly into this environment.

In [10]:
# Dependencies
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install langchain_ollama

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Bringing the server up

In [11]:
# Start Ollama server in the background
import subprocess
import os

# Set the OLLAMA_HOST environment variable to allow connections from the notebook process
os.environ['OLLAMA_HOST'] = '0.0.0.0'

# Start Ollama server as a background process
process = subprocess.Popen(['ollama', 'serve'], preexec_fn=os.setsid, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Ollama server started in the background.")

Ollama server started in the background.


With Ollama now set up and the model pulled, the existing code in the subsequent cells that initializes and uses `ChatOllama` should work as expected.

In [12]:
import time
import requests

# Function to check if Ollama server is ready
def wait_for_ollama(max_retries=30, delay=5):
    for i in range(max_retries):
        try:
            response = requests.get('http://localhost:11434/')
            if response.status_code == 200:
                print("Ollama server is ready.")
                return True
        except requests.exceptions.ConnectionError:
            pass
        print(f"Waiting for Ollama server... ({i+1}/{max_retries})")
        time.sleep(delay)
    print("Ollama server did not become ready.")
    return False

# Wait for the Ollama server to be ready
if wait_for_ollama():
    # Pull the llama3.1:8b model
    print("Pulling llama3.1:8b model. This may take some time...")
    !ollama pull llama3.1:8b
    print("llama3.1:8b model pulled successfully.")
else:
    print("Cannot proceed: Ollama server is not running.")

Ollama server is ready.
Pulling llama3.1:8b model. This may take some time...

llama3.1:8b model pulled successfully.


In [13]:
!nvidia-smi

Sun Apr 19 13:58:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   69C    P0            378W /  600W |   21947MiB /  97887MiB |     87%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Imports

In [14]:
import random
from langchain_ollama import ChatOllama
import pandas as pd

from tqdm import tqdm
import time
import re
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from sklearn.metrics import classification_report

Testing the model with a simplke example

In [15]:
# Initialize the model
# You can adjust 'temperature' for creativity vs. logic
model = ChatOllama(model="llama3.1:8b", temperature=0.7, top_p=0.9)

# Run a simple test
prompt = "Who are you?"
response = model.invoke(prompt)

print(response)

content='I\'m an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."' additional_kwargs={} response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-04-19T13:58:28.836647259Z', 'done': True, 'done_reason': 'stop', 'total_duration': 189789711, 'load_duration': 73662764, 'prompt_eval_count': 14, 'prompt_eval_duration': 9226775, 'eval_count': 23, 'eval_duration': 95160444, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'} id='lc_run--019da609-1e26-7e43-b8c1-4b017ee451f2-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 14, 'output_tokens': 23, 'total_tokens': 37}


In [7]:
df = pd.read_csv("./sampled_50_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4841 entries, 0 to 4840
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ai_target     4841 non-null   int64 
 1   source_model  4841 non-null   object
 2   text100       4841 non-null   object
 3   text200       4841 non-null   object
dtypes: int64(1), object(3)
memory usage: 151.4+ KB


# Trying Vanilla Prompting on the 10% sample - 200 words length data

It is recommended for chat-tuned models like LLama 3.1 to use ***ChatPromptTemplate()*** instead of PromptTemplate.



In [ ]:
system_msg = "You are a classifier. Output ONLY '0' or '1'. No explanation."
user_msg = "Classify this text (0=Human, 1=AI):\n\n{text200}"

prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    ("user", user_msg)
])

chain = prompt | model

texts = df["text200"].tolist()
results = []
start = time.time()

batch_size = 16

for i in tqdm(range(0, len(texts), batch_size), desc="T4 Batch Processing"):
    batch_input = [{"text200": t} for t in texts[i : i + batch_size]]

    # .batch() is much faster on T4 than .invoke() in a loop
    responses = chain.batch(batch_input)

    for res in responses:
        # Clean the output string
        clean_content = res.content.strip()

        # Look for the last digit if the model was chatty
        matches = re.findall(r'\b[01]\b', clean_content)
        if matches:
            # We take the last match as the final answer
            results.append(int(matches[-1]))
        else:
            results.append(-1) # Error flag

df["ai_opinion_200_va"] = results
print(f"Total time: {time.time() - start:.2f}s")

T4 Batch Processing: 100%|██████████| 303/303 [27:03<00:00,  5.36s/it]

Total time: 1623.22s


In [ ]:
y_true = df['ai_target']
y_pred = df['ai_opinion_200_va']

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.50      0.95      0.65      2423
           1       0.38      0.03      0.06      2418

    accuracy                           0.49      4841
   macro avg       0.44      0.49      0.35      4841
weighted avg       0.44      0.49      0.35      4841



### Vanilla Prompting Experiment Results

| Dataset | Model | Temperature | Macro Avg F1-Score | Notes | Time |
| :--- | :--- | :--- | :--- | :--- | :--- |
| sampled_50_200 | llama3.1:8b | 0.7 | 0.43 | lower f1 score for AI class 0.25 compared to 0.61 | 26:59 |
| sampled_50_200 | llama3.1:8b | 0.2 | 0.35 | very bad AI detection with f1-score 0.06  | 27:03 |

# Chain of Thought Prompting - Single Agent


In [18]:
import pandas as pd
import re
from tqdm import tqdm
import time
from langchain_core.prompts import ChatPromptTemplate

df = pd.read_csv("./sampled_50_dataset.csv")

system_msg = "You are a classifier that reasons step by step before giving a final answer."
user_msg = """Is the following text written by a human or AI? Let's think step by step.
Text: {text200}
You must end your response with your final answer on the last line in this exact format:
Final answer: {{0}} for human or {{1}} for AI."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    ("user", user_msg)
])

chain = prompt | model

def parse_response(text):
    match = re.search(r'\{+([01])\}+', text)
    if match:
        return int(match.group(1))
    match = re.search(r'[Ff]inal\s+[Aa]nswer[:\s]+([01])', text)
    if match:
        return int(match.group(1))
    match = re.search(r'([01])\s*(?:\(human\)|\(AI\)|for human|for AI)', text, re.IGNORECASE)
    if match:
        return int(match.group(1))
    match = re.search(r'(?<!\d)([01])(?!\d)(?!.*(?<!\d)[01](?!\d))', text, re.DOTALL)
    if match:
        return int(match.group(1))
    return -1

texts = df["text200"].tolist()
results = []
start = time.time()

for text in tqdm(texts, total=len(texts), desc="CoT Processing"):
    response = chain.invoke({"text200": text})
    ais_thought_response = response.content.strip()
    try:
        ais_numerical_response = parse_response(ais_thought_response)
    except Exception as e:
        print(f"Error: {e}")
        ais_numerical_response = -1
    results.append(ais_numerical_response)

df["ai_opinion_200_cot"] = results
print(f"Total time: {time.time() - start:.2f}s")

CoT Processing: 100%|██████████| 4841/4841 [2:02:56<00:00,  1.52s/it]

Total time: 7376.88s


In [19]:
y_true = df['ai_target']
y_pred = df['ai_opinion_200_cot']

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.48      0.76      0.58      2423
           1       0.40      0.17      0.24      2418

    accuracy                           0.46      4841
   macro avg       0.44      0.46      0.41      4841
weighted avg       0.44      0.46      0.41      4841



In [20]:
df['ai_opinion_200_cot'].value_counts()

,count
ai_opinion_200_cot,
0,3846
1,995


# Step-back prompting

In [21]:
import pandas as pd
import re
from tqdm import tqdm
import time
from langchain_core.prompts import ChatPromptTemplate

df = pd.read_csv("./sampled_50_dataset.csv")

# Step 1: Step-back prompt — asks for general principles
stepback_system = "You are an expert linguist and AI researcher."
stepback_user = "What are the key linguistic and stylistic characteristics that distinguish human-written text from AI-generated text? List the most important principles."

stepback_prompt = ChatPromptTemplate.from_messages([
    ("system", stepback_system),
    ("user", stepback_user)
])

# Step 2: Classification prompt — uses principles + text
classify_system = "You are a text classifier. Use the provided principles to make your classification."
classify_user = """Using these principles for distinguishing human vs AI text:
{principles}

Now classify the following text:
Text: {text200}

Answer with only 0 for human or 1 for AI on the last line in this exact format:
Final answer: {{0}} for human or {{1}} for AI."""

classify_prompt = ChatPromptTemplate.from_messages([
    ("system", classify_system),
    ("user", classify_user)
])

stepback_chain = stepback_prompt | model
classify_chain = classify_prompt | model

def parse_response(text):
    match = re.search(r'\{+([01])\}+', text)
    if match:
        return int(match.group(1))
    match = re.search(r'[Ff]inal\s+[Aa]nswer[:\s]+([01])', text)
    if match:
        return int(match.group(1))
    match = re.search(r'([01])\s*(?:\(human\)|\(AI\)|for human|for AI)', text, re.IGNORECASE)
    if match:
        return int(match.group(1))
    match = re.search(r'(?<!\d)([01])(?!\d)(?!.*(?<!\d)[01](?!\d))', text, re.DOTALL)
    if match:
        return int(match.group(1))
    return -1

texts = df["text200"].tolist()
results = []
start = time.time()

# Get the step-back principles once — no need to call it 5000 times
# since the question is the same every time
principles = stepback_chain.invoke({}).content.strip()
print(f"Step-back principles:\n{principles}\n")

for text in tqdm(texts, total=len(texts), desc="Step-back Processing"):
    response = classify_chain.invoke({"principles": principles, "text200": text})
    ais_response = response.content.strip()
    try:
        ais_numerical_response = parse_response(ais_response)
    except Exception as e:
        print(f"Error: {e}")
        ais_numerical_response = -1
    results.append(ais_numerical_response)

df["ai_opinion_200_stepback"] = results
print(f"Total time: {time.time() - start:.2f}s")

Step-back principles:
As a linguist and AI researcher, I've identified several key characteristics that distinguish human-written text from AI-generated text. Here are the most important principles:

**Linguistic Characteristics:**

1. **Syntax and Sentence Structure**: Human writers tend to use more complex sentence structures, including subordinate clauses, relative clauses, and embedded questions. AI-generated text often relies on simpler sentence structures.
2. **Vocabulary and Word Choice**: Humans choose words that are more nuanced, idiomatic, and context-dependent. AI models may select words based solely on frequency and co-occurrence patterns.
3. **Grammar and Morphology**: Human writers tend to use grammatical structures and morphological features (e.g., verb conjugation, agreement) in a way that's more natural and context-sensitive.
4. **Idiomaticity and Colloquialisms**: Humans often use idioms, colloquial expressions, and figurative language, which can be challenging for AI

Step-back Processing: 100%|██████████| 4841/4841 [2:05:37<00:00,  1.56s/it]

Total time: 7542.31s


In [22]:
y_true = df['ai_target']
y_pred = df['ai_opinion_200_stepback']

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

          -1       0.00      0.00      0.00         0
           0       0.40      0.50      0.45      2423
           1       0.33      0.24      0.28      2418

    accuracy                           0.37      4841
   macro avg       0.25      0.24      0.24      4841
weighted avg       0.37      0.37      0.36      4841



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [23]:
df['ai_opinion_200_stepback'].value_counts()

,count
ai_opinion_200_stepback,
0,2989
1,1702
-1,150


In [24]:
df.to_csv("cot_and_stepback.csv", index=False)